# Evaluation Results Summary

This notebook summarizes the LLM-as-a-Judge evaluation results comparing GPT predictions with annotator annotations.


In [ ]:
import json
import os
from pathlib import Path
from collections import defaultdict
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)


In [ ]:
consolidate_model = 'gpt-5.1'
judge_model = 'gpt-5.1'

In [ ]:
def load_all_results(base_dir="eval_results"):
    """Load all evaluation results from eval_results directory."""
    results = defaultdict(lambda: defaultdict(list))
    
    base_path = Path(base_dir)
    if not base_path.exists():
        print(f"Directory {base_dir} not found!")
        return results
    
    # Iterate through model directories
    for model_dir in base_path.iterdir():
        if not model_dir.is_dir():
            continue
        
        model_name = model_dir.name
        unified_dir = model_dir / "all_at_once_taxonomy" / f"unified_{consolidate_model}"
        
        if not unified_dir.exists():
            continue
        
        # Iterate through task types (mast, whowhen)
        for task_dir in unified_dir.iterdir():
            if not task_dir.is_dir():
                continue
            
            task_type = task_dir.name
            
            # Load all JSON files (excluding run.log)
            for json_file in task_dir.glob("*.json"):
                try:
                    with open(json_file, 'r', encoding='utf-8') as f:
                        data = json.load(f)
                        
                    # Extract comparisons and summary
                    if 'comparisons' in data and 'summary' in data:
                        results[model_name][task_type].append({
                            'file': json_file.name,
                            'comparisons': data['comparisons'],
                            'summary': data['summary'],
                            'total_gpt_steps': data.get('total_gpt_steps', 0),
                            'total_annotator_steps': data.get('total_annotator_steps', 0),
                            'common_steps': data.get('common_steps', 0)
                        })
                except Exception as e:
                    print(f"Error loading {json_file}: {e}")
    
    return results


In [ ]:
# Load all results
all_results = load_all_results(f"eval_results_{judge_model}")

print("Models found:", list(all_results.keys()))
for model in all_results:
    print(f"\n{model}:")
    for task_type in all_results[model]:
        print(f"  {task_type}: {len(all_results[model][task_type])} files")


In [ ]:
def extract_scores(all_results):
    """Extract all scores from results into a flat structure.
    Combines mast and whowhen into 'manual', keeps cap separate."""
    scores = []
    
    for model_name, tasks in all_results.items():
        for task_type, files in tasks.items():
            # Combine mast and whowhen into 'manual', keep cap as is
            grouped_task_type = task_type
            
            for file_data in files:
                for comparison in file_data['comparisons']:
                    if comparison.get('overall_score') is not None:
                        scores.append({
                            'model': model_name,
                            'task_group': grouped_task_type,  # New grouped category
                            'file': file_data['file'],
                            'step': comparison.get('step', ''),
                            'overall_score': comparison.get('overall_score'),
                            'fail_reason_score': comparison.get('fail_reason_score'),
                            'ideal_action_score': comparison.get('ideal_action_score'),
                            'faithfulness_score': comparison.get('faithfulness_score', 0)
                        })
    
    return pd.DataFrame(scores)


In [ ]:
# Extract scores into DataFrame
df_scores = extract_scores(all_results)

## Average Scores by Model and Task Type


In [ ]:
# Create pivot tables for visualization (using task_group: manual = mast+whowhen, cap separate)
pivot_overall = df_scores.pivot_table(
    values='overall_score', 
    index='model', 
    columns='task_group', 
    aggfunc='mean'
).round(2)

print("Average Overall Score by Model and Task Group (manual = mast+whowhen, cap separate):")
print(pivot_overall)
print("\n" + "=" * 80)